In [1]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

from scripts.utils import *
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Rescaling, Dropout ,Input, GlobalAveragePooling2D
from keras.callbacks import TensorBoard, ModelCheckpoint, EarlyStopping
from keras.optimizers import SGD, Adam
from keras import layers
from keras.applications import ResNet50

mixed_precision.set_global_policy("mixed_float16")

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

2026-03-13 22:01:09.470218: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-13 22:01:09.546731: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# directory where images are located
data_dir = "../data/original"
image_size = (96, 96)
batch_size = 64

file_paths = []
labels = []

# the name of each folder corresponds to the label of the images within
class_names = os.listdir(data_dir)
class_to_index = {name: i for i, name in enumerate(class_names)}

# for each folder, we'll save the path of each file with its correspondent label
for class_name in class_names:
    class_dir = f"{data_dir}/{class_name}"
    images = sorted(os.listdir(class_dir))
    for img_path in images:
        file_paths.append(class_dir + "/" + str(img_path))
        labels.append(class_to_index[class_name])

# now we have all the paths and labels together
file_paths = np.array(file_paths)

# bad_files = []

# for path in file_paths:
#     try:
#         if not check_jpeg(path):
#             bad_files.append(path)
#     except:
#         bad_files.append(path)

# print("invalid files:" +  str(len(bad_files)))

# todo: include if

# fix bad_files (already done)
# for path in bad_files:
#     img = Image.open(path)
#     new_path = path.rsplit(".", 1)[0] + ".jpg"
#     img.convert("RGB").save(new_path, "JPEG")

In [3]:
labels = np.array(labels)
dimension = 3

X_train, X_test, y_train, y_test = train_test_split(
    file_paths,
    labels,
    test_size=0.15,
    stratify=labels,
    random_state=42
)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

full_train_X, full_train_y = X_train, y_train

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.15,
    stratify=np.array(y_train),
    random_state=42
)


X_train_full, y_train_full = load_full_dataset(X_train, y_train)
X_train_full = X_train_full.numpy().astype(int).squeeze()
y_train_labels = np.argmax(y_train_full.numpy(), axis = 1)

X_val_full, y_val_full = load_full_dataset(X_val, y_val)
X_val_full = X_val_full.numpy().astype(int).squeeze()
y_val_labels = np.argmax(y_val_full.numpy(), axis = 1)

X_test_full, y_test_full = load_full_dataset(X_test, y_test)
X_test_full = X_test_full.numpy().astype(int).squeeze()
y_test_labels = np.argmax(y_test_full.numpy(), axis = 1)


I0000 00:00:1773435674.688527  124040 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5563 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9
2026-03-13 22:01:18.420444: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-03-13 22:01:22.536639: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [4]:
class_weights = dict(enumerate(class_weights))

## Custom model

In [5]:
def build_custom_model(n_filters = 128, n_neurons = 64, learning_rate = 1e-2, dropout_rate = 0.5, 
                      kernel_size = 3, pool_size = 2, beta_2 = 0.99, sparse = True, momentum = 0.9 ,**kwargs):
    '''
    
    '''
    
    normalization_layer = Rescaling(1./255)
    
    model = Sequential([
    Input(shape = (image_size[0], image_size[1], dimension)),
    normalization_layer,
    Conv2D(n_filters, kernel_size, activation = "relu", padding = "same"),
    MaxPooling2D(pool_size),
    Conv2D(int(n_filters*2), kernel_size, activation = "relu", padding = "same"),
    MaxPooling2D(pool_size),
    GlobalAveragePooling2D(),
    Dense(n_neurons, activation="relu"),
    Dropout(dropout_rate),
    Dense(int(n_neurons/2), activation="relu"),
    Dropout(dropout_rate),
    Dense(10, activation = "softmax")
                    ])
    
    # for the first model only SGD optimizer is used.
    optimizer = Adam(learning_rate=learning_rate, beta_1=momentum, beta_2=beta_2)
    
    loss = "categorical_crossentropy"
    if sparse:
        loss = "sparse_categorical_crossentropy"
    model.compile(loss = loss,
        optimizer = optimizer,
        metrics = ["accuracy"])
    
    return model

In [ ]:
which_arch = "custom"

param_grid = {"learning_rate": [1e-2, 1e-3],
              "kernel_size" : [3, 5],
              "beta_2": [0.99, 0.999],
              "n_filters": [64, 128],
              "n_neurons": [64, 128]}

param_combinations = [
        dict(zip(param_grid.keys(), values))
        for values in itertools.product(*param_grid.values())
    ]

results_tuning = []

history_kept = None
for i, params in enumerate(param_combinations):      
        print(f"Combination: {i+1}")
        print(str(params))
        
        dft_results = pd.DataFrame(params.items()).set_index(0).rename({1: f"Run {i+1}"}, axis = 1).T
        # CLEAR PREVIOUS MODELS
        
        # build model
        model = build_custom_model(**params)
        
        # get id
        # run_id, run_logdir = get_run_logdir(which_arch, f"ip_{i+1}")
        # save results
        
        early_stopping = EarlyStopping(patience = 5, restore_best_weights=True)
        
        model.fit(X_train_full, y_train_labels,
                    validation_data=(X_val_full, y_val_labels), 
                    epochs=15, verbose = 0, 
                    class_weight=class_weights,
                    callbacks= [early_stopping],
                    batch_size=batch_size)
        
        history = model.history.history

        if history_kept is None:
                history_kept = history
        else:
                if history["val_loss"][-1] < val_loss:
                        history_kept = history
                        
        # todo: add other metrics (macro f1, etc etc)
        acc = history["accuracy"][-1]
        val_acc = history["val_accuracy"][-1]
        val_loss = history["val_loss"][-1]
        print(acc, val_acc, val_loss)
                
        dft_results["accuracy"] = acc
        dft_results["val_accuracy"] = val_acc
        dft_results["val_loss"] = val_loss
        
        results_tuning.append(dft_results)
        tf.keras.backend.clear_session()

results_tuning = pd.concat(results_tuning)
results_tuning = pd.DataFrame(results_tuning).sort_values("val_loss", ascending=False)

results_tuning.sort_values("val_loss").to_excel("logs/custom/CV/results_cv.xlsx")
# best_params = results_tuning.astype({"kernel_size": int, "n_filters": int, "n_neurons": int}).sort_values("val_loss").iloc[1, :5].to_dict()
index_best_params = int(results_tuning.reset_index().sort_values("val_loss").iloc[0,0].split("Run ")[-1])-1

Combination: 1
{'learning_rate': 0.01, 'kernel_size': 3, 'beta_2': 0.99, 'n_filters': 64, 'n_neurons': 64}


2026-03-13 20:45:59.834259: I external/local_xla/xla/service/service.cc:163] XLA service 0x7662d000aeb0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-03-13 20:45:59.834286: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2026-03-13 20:45:59.857358: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-03-13 20:46:00.047261: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91700
2026-03-13 20:46:00.187763: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-13 20:46:00.

0.10152442008256912 0.061104584485292435 2.30914306640625
Combination: 2
{'learning_rate': 0.01, 'kernel_size': 3, 'beta_2': 0.99, 'n_filters': 64, 'n_neurons': 128}


2026-03-13 20:46:22.768638: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-13 20:46:22.768698: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-13 20:46:22.768721: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-13 20:46:22.768732: I external/l

0.09105050563812256 0.07461809366941452 2.302311897277832
Combination: 3
{'learning_rate': 0.01, 'kernel_size': 3, 'beta_2': 0.99, 'n_filters': 128, 'n_neurons': 64}


...

0.0882505476474762 0.14864864945411682 2.301499128341675
Combination: 4
{'learning_rate': 0.01, 'kernel_size': 3, 'beta_2': 0.99, 'n_filters': 128, 'n_neurons': 128}


2026-03-13 20:47:04.223744: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-13 20:47:04.223805: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-13 20:47:04.397026: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1188', 4 bytes spill stores, 4 bytes spill loads

2026-03-13 20:47:04.903615: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : R

0.08990977704524994 0.03701527789235115 2.3037564754486084
Combination: 5
{'learning_rate': 0.01, 'kernel_size': 3, 'beta_2': 0.999, 'n_filters': 64, 'n_neurons': 64}


2026-03-13 20:47:23.179533: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-13 20:47:23.179586: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-13 20:47:23.179597: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-13 20:47:23.365947: I external/l

0.08731722831726074 0.11457109451293945 2.3041832447052
Combination: 6
{'learning_rate': 0.01, 'kernel_size': 3, 'beta_2': 0.999, 'n_filters': 64, 'n_neurons': 128}
0.08171730488538742 0.03701527789235115 2.3036413192749023
Combination: 7
{'learning_rate': 0.01, 'kernel_size': 3, 'beta_2': 0.999, 'n_filters': 128, 'n_neurons': 64}
0.08399875462055206 0.06051703914999962 2.3155972957611084
Combination: 8
{'learning_rate': 0.01, 'kernel_size': 3, 'beta_2': 0.999, 'n_filters': 128, 'n_neurons': 128}
0.09758374094963074 0.06051703914999962 2.3063628673553467
Combination: 9
{'learning_rate': 0.01, 'kernel_size': 5, 'beta_2': 0.99, 'n_filters': 64, 'n_neurons': 64}
0.07072487473487854 0.12808459997177124 2.3005456924438477
Combination: 10
{'learning_rate': 0.01, 'kernel_size': 5, 'beta_2': 0.99, 'n_filters': 64, 'n_neurons': 128}
0.10442808270454407 0.03701527789235115 2.315884590148926
Combination: 11
{'learning_rate': 0.01, 'kernel_size': 5, 'beta_2': 0.99, 'n_filters': 128, 'n_neurons': 6

In [ ]:
best_params = param_combinations[index_best_params]
save_params("logs/custom/", "custom_model", best_params)

In [131]:
# one 
for i in range(15):
    model = build_custom_model(**best_params)

    early_stopping = EarlyStopping(patience = 5, restore_best_weights=True)

    model.fit(X_train_full, y_train_labels,
                validation_data=(X_val_full, y_val_labels), 
                epochs=30, verbose = 0,     
                class_weight=class_weights,
                callbacks= [early_stopping],
                batch_size=batch_size)

    pd.DataFrame(model.history.history).to_csv(f"logs/custom/curves_data_{i+1}.csv")

## MobileNetV2

In [5]:
def build_MobileNetV2(learning_rate = 1e-2, momentum = 0.9, beta_2 = 0.99, **kwargs):
    
    normalization_layer = Rescaling(1./255)
    
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(96, 96, 3),
        include_top=False,
        weights="imagenet"
    )

    base_model.trainable = False

    model = tf.keras.models.Sequential([normalization_layer,
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(10, activation="softmax")
    ])

    optimizer = Adam(learning_rate=learning_rate, beta_1=momentum, beta_2=beta_2)
    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    
    return model


In [ ]:
which_arch = "mobilenetv2"

param_grid = {"learning_rate": [1e-2, 1e-3],
              "momentum": [0.0, 0.9],
              "beta_2": [0.99, 0.999]}

param_combinations = [
        dict(zip(param_grid.keys(), values))
        for values in itertools.product(*param_grid.values())
    ]

results_tuning = []

history_kept = None
for i, params in enumerate(param_combinations):      
        print(f"Combination: {i+1}")
        print(str(params))
        
        dft_results = pd.DataFrame(params.items()).set_index(0).rename({1: f"Run {i+1}"}, axis = 1).T
        # CLEAR PREVIOUS MODELS
        
        # build model
        model = build_MobileNetV2(**params)
        
        # get id
        # run_id, run_logdir = get_run_logdir(which_arch, f"ip_{i+1}")
        # save results
        
        early_stopping = EarlyStopping(patience = 5, restore_best_weights=True)
        
        model.fit(X_train_full, y_train_labels,
                    validation_data=(X_val_full, y_val_labels), 
                    epochs=15, verbose = 0, 
                    class_weight=class_weights,
                    callbacks= [early_stopping],
                    batch_size=batch_size)
        
        history = model.history.history

        if history_kept is None:
                history_kept = history
        else:
                if history["val_loss"][-1] < val_loss:
                        history_kept = history
                        
        # todo: add other metrics (macro f1, etc etc)
        acc = history["accuracy"][-1]
        val_acc = history["val_accuracy"][-1]
        val_loss = history["val_loss"][-1]
        print(acc, val_acc, val_loss)
                
        dft_results["accuracy"] = acc
        dft_results["val_accuracy"] = val_acc
        dft_results["val_loss"] = val_loss
        
        results_tuning.append(dft_results)
        tf.keras.backend.clear_session()

results_tuning = pd.concat(results_tuning)
results_tuning = pd.DataFrame(results_tuning).sort_values("val_loss", ascending=False)

Combination: 1
{'learning_rate': 0.01, 'momentum': 0.0, 'beta_2': 0.99}


2026-03-13 22:01:31.662827: I external/local_xla/xla/service/service.cc:163] XLA service 0x746d800024b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-03-13 22:01:31.662861: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2026-03-13 22:01:31.753342: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-03-13 22:01:32.313581: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91700
2026-03-13 22:01:45.765691: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng45{k2=4,k5=3,k14=3} for conv (f16[64,6,6,384]{3,2,1,0}, u8[0]{0}) custom-call(f16[64,6,6,64]{3,2,1,0}, f16[384,1,1,64]{3,2,1,0}), window={size=1x1}, dim_labels=b01f_o01i->b01f, custom_call_target="__cudnn$convForward", backend_conf

0.8990978002548218 0.8343125581741333 1.1012667417526245
Combination: 2
{'learning_rate': 0.01, 'momentum': 0.0, 'beta_2': 0.999}


2026-03-13 22:03:05.887458: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng3{k11=0} for conv (f32[43,192,12,12]{3,2,1,0}, u8[0]{0}) custom-call(f32[43,192,12,12]{3,2,1,0}, f32[192,1,3,3]{3,2,1,0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, feature_group_count=192, custom_call_target="__cudnn$convForward", backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reification_cost":[]} is taking a while...
2026-03-13 22:03:05.896692: E external/local_xla/xla/service/slow_operation_alarm.cc:140] The operation took 6.134178959s
Trying algorithm eng3{k11=0} for conv (f32[43,192,12,12]{3,2,1,0}, u8[0]{0}) custom-call(f32[43,192,12,12]{3,2,1,0}, f32[192,1,3,3]{3,2,1,0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, feature_group_count=192, custom_call_target="__cudnn

0.9184901118278503 0.8202115297317505 1.0628385543823242
Combination: 3
{'learning_rate': 0.01, 'momentum': 0.9, 'beta_2': 0.99}
0.9241937398910522 0.8413631319999695 1.0553761720657349
Combination: 4
{'learning_rate': 0.01, 'momentum': 0.9, 'beta_2': 0.999}
0.9273048043251038 0.8354876637458801 0.9197680950164795
Combination: 5
{'learning_rate': 0.001, 'momentum': 0.0, 'beta_2': 0.99}
0.9626672267913818 0.8560516834259033 0.48674243688583374
Combination: 6
{'learning_rate': 0.001, 'momentum': 0.0, 'beta_2': 0.999}
0.9588302373886108 0.8607520461082458 0.4503916800022125
Combination: 7
{'learning_rate': 0.001, 'momentum': 0.9, 'beta_2': 0.99}
0.9508451819419861 0.8566392660140991 0.45384231209754944
Combination: 8
{'learning_rate': 0.001, 'momentum': 0.9, 'beta_2': 0.999}
0.9567561745643616 0.858401894569397 0.4416206181049347


OSError: Cannot save file into a non-existent directory: 'logs/mobilenetv2/CV'

In [7]:

results_tuning.sort_values("val_loss").to_excel(f"logs/{which_arch}/CV/results_cv.xlsx")
# best_params = results_tuning.astype({"kernel_size": int, "n_filters": int, "n_neurons": int}).sort_values("val_loss").iloc[1, :5].to_dict()
index_best_params = int(results_tuning.reset_index().sort_values("val_loss").iloc[0,0].split("Run ")[-1])-1

In [8]:
best_params = param_combinations[index_best_params]
save_params(f"logs/{which_arch}/", f"{which_arch}_model", best_params)

In [9]:
# one 
for i in range(15):
    model = build_MobileNetV2(**best_params)

    early_stopping = EarlyStopping(patience = 5, restore_best_weights=True)

    model.fit(X_train_full, y_train_labels,
                validation_data=(X_val_full, y_val_labels), 
                epochs=30, verbose = 0,     
                class_weight=class_weights,
                callbacks= [early_stopping],
                batch_size=batch_size)

    pd.DataFrame(model.history.history).to_csv(f"logs/{which_arch}/curves_data_{i+1}.csv")